<a href="https://colab.research.google.com/github/lolxd23/Baruch-College/blob/main/aiXsustaniblity.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
'''
help create an application that can be linked to existing building electricity, gas account to analyze energy usage and provide suggestions to to reduce consumption
it should have a way to. plug in building condition/ age/ products used etc.
include weather pridictibily to combat storm situations.
'''

SyntaxError: invalid syntax (3449548096.py, line 1)

In [9]:
# analyze data
# Novemeber - Decemeber

In [ ]:
# ============================================================
# WattWise — single-cell version. Paste this WHOLE thing into
# one notebook cell and run itq1. No import needed.
# Just change CSV_FILE below to your file's name or full path.
# ============================================================

CSV_FILE = "coned - Sheet10.csv"   # <-- change to your path if needed

import numpy as np
import pandas as pd
from dataclasses import dataclass
import matplotlib.pyplot as plt


@dataclass
class RatePlan:
    flat_rate: float = 0.40
    fixed_monthly: float = 22.31
    tou_peak_rate: float = 0.55
    tou_offpeak_rate: float = 0.27
    peak_hours: tuple = (8, 24)
    co2_lb_per_kwh: float = 0.39


def load_greenbutton(path):
    raw = pd.read_csv(path)
    iv = raw[["TYPE", "DATE", "START TIME", "END TIME", "USAGE (kWh)"]].copy()
    iv = iv.dropna(subset=["USAGE (kWh)"])
    iv["kwh"] = pd.to_numeric(iv["USAGE (kWh)"], errors="coerce")
    iv["ts"] = pd.to_datetime(iv["DATE"] + " " + iv["START TIME"],
                              format="%Y-%m-%d %H:%M", errors="coerce")
    iv = iv.dropna(subset=["ts", "kwh"]).sort_values("ts").reset_index(drop=True)
    iv["hour"] = iv["ts"].dt.hour
    iv["dow"] = iv["ts"].dt.dayofweek
    iv["date"] = iv["ts"].dt.date
    iv["is_weekend"] = iv["dow"] >= 5
    iv["kw"] = iv["kwh"] / 0.25
    fuel = iv["TYPE"].iloc[0] if len(iv) else "Electric usage"
    daily = raw[["DATE.1", "Total Usage", "Total Cost"]].copy()
    daily.columns = ["date", "kwh", "cost"]
    daily = daily.dropna(subset=["kwh"])
    daily["kwh"] = pd.to_numeric(daily["kwh"], errors="coerce")
    daily["cost"] = pd.to_numeric(daily["cost"], errors="coerce")
    daily["date"] = pd.to_datetime(daily["date"], errors="coerce")
    daily = daily.dropna(subset=["date", "kwh"]).sort_values("date")
    return {"intervals": iv, "daily": daily, "fuel": fuel}


class EnergyAnalyzer:
    def __init__(self, data, rate=None):
        self.iv = data["intervals"]; self.daily = data["daily"]
        self.fuel = data.get("fuel", "Electric"); self.rate = rate or RatePlan()
        self.n_days = self.iv["date"].nunique()

    def always_on_load(self):
        return float(self.iv.groupby("date")["kw"].min().median())

    def hourly_profile(self):
        return self.iv.groupby("hour")["kwh"].mean() * 4

    def weekday_weekend_profile(self):
        wk = self.iv[~self.iv.is_weekend].groupby("hour")["kwh"].mean() * 4
        we = self.iv[self.iv.is_weekend].groupby("hour")["kwh"].mean() * 4
        return wk, we

    def peak_demand(self):
        i = self.iv.loc[self.iv["kw"].idxmax()]
        return {"kw": round(float(i.kw), 2), "when": i.ts}

    def peak_offpeak_split(self):
        lo, hi = self.rate.peak_hours
        m = (self.iv.hour >= lo) & (self.iv.hour < hi)
        return self.iv.loc[m, "kwh"].sum(), self.iv.loc[~m, "kwh"].sum()

    def heating_trend(self):
        d = self.daily.copy(); d["t"] = (d["date"] - d["date"].min()).dt.days
        return float(np.polyfit(d["t"], d["kwh"], 1)[0]) if len(d) >= 3 else 0.0

    def anomalies(self, z=2.0):
        d = self.daily["kwh"]
        return self.daily[self.daily["kwh"] > d.mean() + z * d.std()]

    def summary(self):
        total_kwh = self.daily["kwh"].sum(); total_cost = self.daily["cost"].sum()
        ao = self.always_on_load()
        return {"fuel": self.fuel, "days": self.n_days, "total_kwh": total_kwh,
                "total_cost": total_cost, "avg_daily_kwh": total_kwh / self.n_days,
                "avg_daily_cost": total_cost / self.n_days,
                "annual_kwh_proj": total_kwh / self.n_days * 365,
                "annual_cost_proj": total_cost / self.n_days * 365,
                "always_on_kw": ao,
                "always_on_share": ao * 24 * self.n_days / total_kwh,
                "peak": self.peak_demand(), "heating_slope": self.heating_trend()}


@dataclass
class Suggestion:
    title: str; detail: str
    annual_kwh_saved: float; annual_dollars_saved: float
    @property
    def co2(self): return self.annual_kwh_saved * RatePlan().co2_lb_per_kwh


class Recommender:
    def __init__(self, a): self.a = a; self.r = a.rate
    def generate(self):
        s = self.a.summary(); out = []; annual = s["annual_kwh_proj"]
        ao_kwh = s["always_on_kw"] * 24 * 365; cut = ao_kwh * 0.30
        out.append(Suggestion("Cut always-on / phantom load",
            f"Your home never drops below {s['always_on_kw']*1000:.0f} W — that floor "
            f"is ~{s['always_on_share']*100:.0f}% of all usage. Smart strips + unplugging "
            f"idle electronics trim ~30%.", cut, cut * self.r.flat_rate))
        peak, off = self.a.peak_offpeak_split(); ps = peak / (peak + off)
        flat = annual * self.r.flat_rate
        tou_shift = annual * ((ps - 0.15) * self.r.tou_peak_rate +
                              (1 - ps + 0.15) * self.r.tou_offpeak_rate)
        if flat - tou_shift > 0:
            out.append(Suggestion("Shift use off-peak + time-of-use rate",
                f"{ps*100:.0f}% of energy is used in peak hours "
                f"({self.r.peak_hours[0]}:00-{self.r.peak_hours[1]:02d}:00). Run "
                f"dishwasher/laundry/EV overnight on ConEd TOU.", 0, flat - tou_shift))
        if s["heating_slope"] > 0.05:
            save = s["heating_slope"] * 90 * 0.20
            out.append(Suggestion("Tame the winter heating climb",
                f"Daily usage rises ~{s['heating_slope']:.2f} kWh/day as it cools — "
                f"heating load. Smart thermostat setback + weather-stripping help.",
                save, save * self.r.flat_rate))
        anom = self.a.anomalies()
        if len(anom):
            days = ", ".join(anom["date"].dt.strftime("%b %d").tolist()[:4])
            out.append(Suggestion("Investigate spike days",
                f"{len(anom)} day(s) ran well above norm ({days}). Check for space "
                f"heaters left on, guests, or a failing appliance.", 0, 0))
        led = annual * 0.05
        out.append(Suggestion("LED lighting + ENERGY STAR upgrades",
            "Swapping bulbs to LED and aging appliances to ENERGY STAR trims ~5%.",
            led, led * self.r.flat_rate))
        out.sort(key=lambda x: x.annual_dollars_saved, reverse=True)
        return out


def make_plots(a, path="energy_app_plots.png"):
    fig, ax = plt.subplots(2, 2, figsize=(13, 9))
    fig.suptitle("WattWise — Energy Account Analysis", fontsize=15, fontweight="bold")
    prof = a.hourly_profile()
    ax[0, 0].bar(prof.index, prof.values, color="#185FA5")
    ax[0, 0].axhline(a.always_on_load(), color="#D85A30", ls="--",
                     label=f"always-on ({a.always_on_load()*1000:.0f} W)")
    ax[0, 0].set_title("Average load shape by hour"); ax[0, 0].legend(); ax[0, 0].grid(alpha=.3)
    wk, we = a.weekday_weekend_profile()
    ax[0, 1].plot(wk.index, wk.values, "o-", label="weekday", color="#534AB7")
    ax[0, 1].plot(we.index, we.values, "s-", label="weekend", color="#1D9E75")
    ax[0, 1].set_title("Weekday vs weekend"); ax[0, 1].legend(); ax[0, 1].grid(alpha=.3)
    d = a.daily
    ax[1, 0].plot(d["date"], d["kwh"], "o-", color="#185FA5")
    z = np.polyfit(range(len(d)), d["kwh"], 1)
    ax[1, 0].plot(d["date"], np.poly1d(z)(range(len(d))), "--", color="#D85A30",
                  label=f"trend {z[0]:+.2f} kWh/day")
    ax[1, 0].set_title("Daily usage over time"); ax[1, 0].legend(); ax[1, 0].grid(alpha=.3)
    ax[1, 0].tick_params(axis="x", rotation=45)
    ax[1, 1].hist(d["kwh"], bins=15, color="#534AB7", alpha=.8)
    ax[1, 1].axvline(d["kwh"].mean(), color="#D85A30", ls="--", label="mean")
    ax[1, 1].set_title("Distribution of daily usage"); ax[1, 1].legend(); ax[1, 1].grid(alpha=.3)
    plt.tight_layout(rect=[0, 0, 1, 0.96]); plt.savefig(path, dpi=120); plt.show()
    return path


def run(path):
    data = load_greenbutton(path); a = EnergyAnalyzer(data); s = a.summary()
    print("\n" + "#" * 64)
    print("#  WattWise — Energy Account Analysis")
    print("#" * 64)
    print(f"\n  Account fuel : {s['fuel']}")
    print(f"  Period       : {s['days']} days of interval data")
    print(f"  Total usage  : {s['total_kwh']:,.0f} kWh  (${s['total_cost']:,.2f})")
    print(f"  Daily average: {s['avg_daily_kwh']:.1f} kWh/day (${s['avg_daily_cost']:.2f}/day)")
    print(f"  ANNUALIZED   : {s['annual_kwh_proj']:,.0f} kWh ~${s['annual_cost_proj']:,.0f}/yr")
    print(f"\n  Always-on load : {s['always_on_kw']*1000:.0f} W 24/7 "
          f"= {s['always_on_share']*100:.0f}% of everything")
    print(f"  Peak demand    : {s['peak']['kw']:.2f} kW at {s['peak']['when']:%b %d, %I:%M %p}")
    print(f"  Heating trend  : {s['heating_slope']:+.2f} kWh/day as it cools")
    recs = Recommender(a).generate()
    print("\n  " + "-" * 60)
    print("  RECOMMENDATIONS (ranked by annual $ saved):")
    print("  " + "-" * 60)
    tot = 0.0
    for i, r in enumerate(recs, 1):
        print(f"\n  {i}. {r.title}\n     {r.detail}")
        if r.annual_dollars_saved > 0:
            print(f"     -> save ~${r.annual_dollars_saved:,.0f}/yr"
                  + (f"  ({r.annual_kwh_saved:,.0f} kWh, {r.co2:,.0f} lb CO2)"
                     if r.annual_kwh_saved else ""))
        tot += r.annual_dollars_saved
    print("\n  " + "-" * 60)
    print(f"  TOTAL OPPORTUNITY: ~${tot:,.0f}/yr ({tot/s['annual_cost_proj']*100:.0f}% of bill)")
    print("  " + "-" * 60)
    make_plots(a)
    return a, recs


analyzer, recommendations = run(CSV_FILE)


################################################################
#  WattWise — Energy Account Analysis
################################################################

  Account fuel : Electric usage
  Period       : 56 days of interval data
  Total usage  : 715 kWh  ($286.16)
  Daily average: 12.8 kWh/day ($5.11/day)
  ANNUALIZED   : 4,663 kWh ~$1,865/yr

  Always-on load : 160 W 24/7 = 30% of everything
  Peak demand    : 4.92 kW at Nov 16, 08:00 PM
  Heating trend  : +0.03 kWh/day as it cools

  ------------------------------------------------------------
  RECOMMENDATIONS (ranked by annual $ saved):
  ------------------------------------------------------------

  1. Cut always-on / phantom load
     Your home never drops below 160 W — that floor is ~30% of all usage. Smart strips + unplugging idle electronics trim ~30%.
     -> save ~$168/yr  (420 kWh, 164 lb CO2)

  2. LED lighting + ENERGY STAR upgrades
     Swapping bulbs to LED and aging appliances to ENERGY STAR trims ~5%.


In [ ]:
# ============================================================
# PRISM Change-Point Model — single-cell version.
# Paste this WHOLE thing into ONE notebook cell and run it.
# No import needed. Change CSV_FILE if your file is elsewhere.
# ============================================================

CSV_FILE = "coned - Sheet10.csv"   # <-- change to your path if needed
NOAA_TOKEN = None                  # <-- paste your free NOAA token here for real temps

import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score
import matplotlib.pyplot as plt


# ---- inline CSV loader (so we don't import energy_app) -------------------
def load_daily(path):
    """Read the daily-summary block (date, kwh, cost) from the Green Button CSV."""
    raw = pd.read_csv(path)
    daily = raw[["DATE.1", "Total Usage", "Total Cost"]].copy()
    daily.columns = ["date", "kwh", "cost"]
    daily = daily.dropna(subset=["kwh"])
    daily["kwh"] = pd.to_numeric(daily["kwh"], errors="coerce")
    daily["cost"] = pd.to_numeric(daily["cost"], errors="coerce")
    daily["date"] = pd.to_datetime(daily["date"], errors="coerce")
    return daily.dropna(subset=["date", "kwh"]).sort_values("date").reset_index(drop=True)


# ---- NOAA fetcher (runs locally with a free token) -----------------------
# Register: https://www.ncdc.noaa.gov/cdo-web/token
# NYC Central Park station: GHCND:USW00094728
def fetch_noaa_daily_temps(token, start, end, station="GHCND:USW00094728"):
    import requests
    url = "https://www.ncei.noaa.gov/cdo-web/api/v2/data"
    params = {"datasetid": "GHCND", "stationid": station, "datatypeid": "TAVG",
              "startdate": start, "enddate": end, "units": "standard", "limit": 1000}
    r = requests.get(url, headers={"token": token}, params=params, timeout=30)
    r.raise_for_status()
    df = pd.DataFrame(r.json().get("results", []))
    df["date"] = pd.to_datetime(df["date"])
    return df[["date", "value"]].rename(columns={"value": "temp_f"})


def _demo_nyc_temps(dates):
    """Deterministic stand-in NYC daily temps so the demo runs without NOAA."""
    doy = pd.to_datetime(dates).dt.dayofyear.to_numpy()
    base = 52 - 22 * np.cos(2 * np.pi * (doy - 15) / 365)
    rng = np.random.default_rng(7)
    return pd.Series(base + rng.normal(0, 3, size=len(doy)), index=dates.index)


# ---- the change-point model ----------------------------------------------
class ChangePointModel:
    def __init__(self, bp_grid=range(45, 70)):
        self.bp_grid = list(bp_grid); self.fitted = False

    def fit(self, temp_f, kwh_per_day):
        temp_f = np.asarray(temp_f, float); kwh = np.asarray(kwh_per_day, float)
        best = None
        for bp in self.bp_grid:
            hdd = np.maximum(bp - temp_f, 0)
            cdd = np.maximum(temp_f - bp, 0)
            X = np.column_stack([hdd, cdd])
            lr = LinearRegression().fit(X, kwh)
            r2 = r2_score(kwh, lr.predict(X))
            if best is None or r2 > best["r2"]:
                best = {"bp": bp, "r2": r2, "baseload": lr.intercept_,
                        "heating_slope": lr.coef_[0], "cooling_slope": lr.coef_[1],
                        "model": lr}
        self.__dict__.update(best); self.fitted = True
        return self

    def predict(self, temp_f):
        temp_f = np.asarray(temp_f, float)
        hdd = np.maximum(self.bp - temp_f, 0); cdd = np.maximum(temp_f - self.bp, 0)
        return self.model.predict(np.column_stack([hdd, cdd]))

    def diagnose(self):
        hs = self.heating_slope
        if hs < 0.3:
            return "TIGHT envelope — low heating sensitivity, well insulated."
        elif hs < 0.8:
            return "TYPICAL envelope — moderate insulation."
        return ("LEAKY envelope — high heating slope. Insulation & "
                "air-sealing are the highest-value retrofit.")

    def report(self):
        print("\n" + "=" * 64)
        print("  CHANGE-POINT (PRISM) MODEL — weather-normalized fit")
        print("=" * 64)
        print(f"  Balance-point temperature : {self.bp} F")
        print(f"  Baseload (weather-indep.) : {self.baseload:.2f} kWh/day")
        print(f"  Heating slope             : {self.heating_slope:.3f} kWh/HDD")
        print(f"  Cooling slope             : {self.cooling_slope:.3f} kWh/CDD")
        print(f"  Fit quality (R2)          : {self.r2:.3f}")
        print(f"\n  Diagnosis: {self.diagnose()}")


def plot_fit(model, temp_f, kwh):
    grid = np.linspace(min(temp_f) - 2, max(temp_f) + 2, 100)
    plt.figure(figsize=(9, 6))
    plt.scatter(temp_f, kwh, s=30, alpha=0.6, color="#185FA5", label="daily data")
    plt.plot(grid, model.predict(grid), color="#D85A30", lw=2.5,
             label=f"change-point fit (bp={model.bp}F)")
    plt.axvline(model.bp, color="#1D9E75", ls="--", alpha=0.7,
                label=f"balance point {model.bp}F")
    plt.title("Change-point energy signature")
    plt.xlabel("Daily mean temperature (F)"); plt.ylabel("Daily usage (kWh)")
    plt.legend(); plt.grid(alpha=0.3); plt.tight_layout(); plt.show()


def validate_on_synthetic():
    rng = np.random.default_rng(1)
    temp = rng.uniform(15, 75, 120)
    kwh = 14.0 + 0.9 * np.maximum(60 - temp, 0) + rng.normal(0, 2, 120)
    m = ChangePointModel().fit(temp, kwh)
    print("\n" + "=" * 64)
    print("  VALIDATION — recovering a known electrically-heated signature")
    print("=" * 64)
    print(f"  planted  : bp=60F  baseload=14.0  slope=0.90")
    print(f"  recovered: bp={m.bp}F  baseload={m.baseload:.1f}  "
          f"slope={m.heating_slope:.2f}   R2={m.r2:.3f}")
    print(f"  -> {m.diagnose()}")


def run(csv_path=CSV_FILE, noaa_token=NOAA_TOKEN):
    daily = load_daily(csv_path)

    if noaa_token:
        temps = fetch_noaa_daily_temps(noaa_token,
                                       daily["date"].min().strftime("%Y-%m-%d"),
                                       daily["date"].max().strftime("%Y-%m-%d"))
        daily = daily.merge(temps, on="date", how="left")
        daily["temp_f"] = daily["temp_f"].interpolate()
        src = "NOAA Central Park"
    else:
        daily["temp_f"] = _demo_nyc_temps(daily["date"])
        src = "DEMO temps (set NOAA_TOKEN above for real temperatures)"

    validate_on_synthetic()

    print(f"\n  Temperature source: {src}")
    model = ChangePointModel().fit(daily["temp_f"].to_numpy(), daily["kwh"].to_numpy())
    model.report()
    print(f"\n  Annual baseload floor: {model.baseload*365:,.0f} kWh/yr "
          f"(the part no thermostat change will fix)")
    if model.r2 < 0.2:
        print("\n  INTERPRETATION: electric load here is weather-INSENSITIVE")
        print("  (low slope, low R2). This account is almost certainly gas-heated")
        print("  — so the lever is appliances & phantom load, NOT insulation.")
    plot_fit(model, daily["temp_f"].to_numpy(), daily["kwh"].to_numpy())
    return model


model = run()



  VALIDATION — recovering a known electrically-heated signature
  planted  : bp=60F  baseload=14.0  slope=0.90
  recovered: bp=59F  baseload=14.3  slope=0.91   R2=0.978
  -> LEAKY envelope — high heating slope. Insulation & air-sealing are the highest-value retrofit.

  Temperature source: DEMO temps (set NOAA_TOKEN above for real temperatures)

  CHANGE-POINT (PRISM) MODEL — weather-normalized fit
  Balance-point temperature : 45 F
  Baseload (weather-indep.) : 11.76 kWh/day
  Heating slope             : 0.109 kWh/HDD
  Cooling slope             : 0.881 kWh/CDD
  Fit quality (R2)          : 0.032

  Diagnosis: TIGHT envelope — low heating sensitivity, well insulated.

  Annual baseload floor: 4,293 kWh/yr (the part no thermostat change will fix)

  INTERPRETATION: electric load here is weather-INSENSITIVE
  (low slope, low R2). This account is almost certainly gas-heated
  — so the lever is appliances & phantom load, NOT insulation.
